In [ ]:
import getFamaFrenchFactors as gff

# Get the Fama French 3 factor model (monthly data)
df_ff3_monthly = gff.famaFrench3Factor(frequency='m') 

# Get the Fama French 3 factor model (annual data)
df_ff3_annual = gff.famaFrench3Factor(frequency='a')

In [ ]:
df_ff3_monthly

In [ ]:
# df_ff3_monthly.index = df_ff3_monthly.date_ff_factors


In [ ]:
df_ff3_monthly.rename(columns={"date_ff_factors": 'date'}, inplace=True)
df_ff3_monthly.set_index('date', inplace=True)
df_ff3_monthly.index = pd.to_datetime(df_ff3_monthly.index, format = '%m/%d/%Y').strftime('%Y%m%d')

In [ ]:
df_ff3_monthly.to_csv("../CSV/FamaFrenchSPY.csv")

In [ ]:
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import getFamaFrenchFactors as gff

ticker = 'SPY'
start = '2020-05-01'
end = '2022-08-01'

stock_data = yf.download(ticker, start, end, adjusted=True)

ff3_monthly = gff.famaFrench3Factor(frequency='m')#famaFrench3Factor(frequency='m')
ff3_monthly.rename(columns={"date_ff_factors": 'Date'}, inplace=True)
ff3_monthly.set_index('Date', inplace=True)

In [ ]:
stock_data

In [ ]:
stock_returns = stock_data['Adj Close'].resample('M').last().pct_change().dropna()
stock_returns.name = "Month_Rtn"
ff_data = ff3_monthly.merge(stock_returns,on='Date')

In [ ]:
ff_data

In [ ]:
ff_data.index = pd.to_datetime(ff_data.index, format = '%m/%d/%Y').strftime('%Y%m%d')

In [ ]:
ff_data.to_csv("FamaFrenchSPY.csv")

In [ ]:
X = ff_data[['Mkt-RF', 'SMB', 'HML']]
y = ff_data['Month_Rtn'] - ff_data['RF']
X = sm.add_constant(X)
ff_model = sm.OLS(y, X).fit()
print(ff_model.summary())
intercept, b1, b2, b3 = ff_model.params

In [ ]:
rf = ff_data['RF'].mean()
market_premium = ff3_monthly['Mkt-RF'].mean()
size_premium = ff3_monthly['SMB'].mean()
value_premium = ff3_monthly['HML'].mean()

expected_monthly_return = rf + b1 * market_premium + b2 * size_premium + b3 * value_premium 
expected_yearly_return = expected_monthly_return * 12
print("Expected yearly return: " + str(expected_yearly_return))